# 🧠 NeuroEvasion — GPU Training on Google Colab

> **Train both DQN agents on a free T4 GPU, then download the weights to your local machine.**

## Workflow
1. Upload your codebase (this notebook handles it)
2. Train on GPU (~10-50x faster than CPU)
3. Download the trained `.pt` checkpoint files
4. Load them locally with `python main.py demo --snake-model ... --bait-model ...`

---

## Step 1 — Verify GPU is Available

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU Available: {gpu_name} ({gpu_mem:.1f} GB)")
    print(f"   PyTorch: {torch.__version__}")
    print(f"   CUDA: {torch.version.cuda}")
else:
    print("⚠️  No GPU detected!")
    print("   Go to: Runtime → Change runtime type → GPU (T4)")

## Step 2 — Upload Your Codebase

**Option A (Recommended):** Upload a zip of your `NeuroEvasion` folder.

**Option B:** Clone from GitHub if you've pushed it.

In [ ]:
# === OPTION A: Upload a zip file ===
# 1. On your Mac, run: cd ~/Downloads && zip -r NeuroEvasion.zip NeuroEvasion/ -x '*.venv/*' '*.git/*' '*__pycache__/*'
# 2. Then upload it here:

from google.colab import files
import zipfile, os

print("📁 Upload your NeuroEvasion.zip file...")
uploaded = files.upload()

for filename in uploaded:
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as z:
            z.extractall('.')
        print(f"✅ Extracted {filename}")

# Navigate into the project
os.chdir('/content/NeuroEvasion')
print(f"📂 Working directory: {os.getcwd()}")
!ls -la

In [ ]:
# === OPTION B: Clone from GitHub ===
# Uncomment and edit the line below if you prefer:

# !git clone https://github.com/YOUR_USERNAME/NeuroEvasion.git
# import os; os.chdir('/content/NeuroEvasion')

## Step 3 — Install Dependencies

In [ ]:
# PyTorch is pre-installed on Colab with CUDA support
# We only need pygame (for the renderer module import) and tensorboard
!pip install pygame tensorboard -q
print("✅ Dependencies installed")

## Step 4 — Quick Sanity Check

Run a 10-episode test to verify everything works on GPU before full training.

In [ ]:
# Quick import test
import sys
sys.path.insert(0, '/content/NeuroEvasion')

from config import Config
from environment.env import NeuroEvasionEnv
from agents.dqn_agent import DQNAgent

config = Config()
env = NeuroEvasionEnv(config)

# Test GPU agent creation
agent = DQNAgent(
    in_channels=env.obs_channels,
    grid_size=config.game.grid_size,
    num_actions=env.snake_num_actions,
    config=config.agent,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

device = next(agent.policy_net.parameters()).device
print(f"✅ Agent created on: {device}")

# Quick forward pass test
snake_obs, bait_obs = env.reset()
action = agent.select_action(snake_obs)
print(f"✅ Forward pass works! Selected action: {action}")
print(f"   Observation shape: {snake_obs.shape}")
print(f"   Model parameters: {sum(p.numel() for p in agent.policy_net.parameters()):,}")

## Step 5 — 🚀 Full Training

Now run the actual training. Adjust `--episodes` based on how much time you have:

| Episodes | Approx. Time (T4 GPU) | Quality |
|----------|-----------------------|---------|
| 10,000 | ~5-10 min | Basic behaviors emerge |
| 50,000 | ~30-60 min | Decent strategies |
| 200,000 | ~2-4 hours | Strong co-evolution |
| 500,000 | ~6-10 hours | Full convergence |

> **Tip:** Start with 50,000 to see results, then increase if needed.

In [ ]:
# ==========================================
#   CONFIGURE YOUR TRAINING RUN HERE
# ==========================================

EPISODES = 50_000       # Adjust based on your time budget
GRID_SIZE = 20          # Default: 20×20
LEARNING_RATE = 1e-4    # Default: 0.0001
SEED = 42               # For reproducibility

# ==========================================

!python main.py train \
  --episodes {EPISODES} \
  --grid-size {GRID_SIZE} \
  --lr {LEARNING_RATE} \
  --device cuda \
  --seed {SEED}

## Step 6 — Verify Training Output

In [ ]:
import os

print("📁 Saved checkpoints:")
for f in sorted(os.listdir('checkpoints')):
    if f.endswith('.pt'):
        size_mb = os.path.getsize(f'checkpoints/{f}') / 1e6
        print(f"   {f:40s} ({size_mb:.1f} MB)")

print("\n📁 Training logs:")
for d in os.listdir('logs'):
    log_path = f'logs/{d}'
    if os.path.isdir(log_path):
        n_files = len(os.listdir(log_path))
        print(f"   {d}/ ({n_files} files)")

## Step 7 — Quick Evaluation (on Colab)

In [ ]:
!python main.py evaluate \
  --snake-model checkpoints/snake_final.pt \
  --bait-model checkpoints/bait_final.pt \
  --num-games 500

## Step 8 — 📥 Download Trained Weights

Download the final model weights to your local machine.
Then load them with:
```bash
python main.py demo --snake-model snake_final.pt --bait-model bait_final.pt
```

In [ ]:
from google.colab import files

# Download the final trained models
print("📥 Downloading snake_final.pt...")
files.download('checkpoints/snake_final.pt')

print("📥 Downloading bait_final.pt...")
files.download('checkpoints/bait_final.pt')

In [ ]:
# OPTIONAL: Download ALL checkpoints as a zip
import shutil
from google.colab import files

shutil.make_archive('neuroevasion_checkpoints', 'zip', '.', 'checkpoints')
files.download('neuroevasion_checkpoints.zip')
print("✅ All checkpoints downloaded as zip")

## Step 9 — (Optional) View TensorBoard in Colab

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/

---

## 🏠 Back on Your Local Machine

After downloading the `.pt` files, place them in your `NeuroEvasion/checkpoints/` folder and run:

```bash
cd NeuroEvasion
source .venv/bin/activate

# Watch the trained agents play!
python main.py demo \
  --snake-model checkpoints/snake_final.pt \
  --bait-model checkpoints/bait_final.pt \
  --speed 8

# Evaluate performance
python main.py evaluate \
  --snake-model checkpoints/snake_final.pt \
  --bait-model checkpoints/bait_final.pt \
  --num-games 1000
```

> **Note:** The demo and evaluation commands run on CPU and are lightweight — they don't train, they only do forward passes through the network.